# Training (Spark ML, GCP YARN)
Trains XGB model on baseline features and Holt-Winters features.
Outputs are saved to HDFS under /user/tiennd3886.

In [1]:
import pandas as pd
from datetime import datetime
from pyspark.sql import SparkSession, Window, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator

BASE_HDFS = "/user/tiennd3886"
FEATURE_PATH = f"{BASE_HDFS}/feature_engineering/demand_prediction_features_30m_enhanced"
OUT_BASE_ROOT = f"{BASE_HDFS}/results/sparkml"
MODEL_BASE = f"{BASE_HDFS}/models/sparkml"
TARGET_COL = "pickup_demand_t1"

feature_cols_base = [
    "hour", "dow", "month", "is_weekend",
    "is_holiday", "is_covid_lockdown", "is_covid_partial",
    "lag_6", "lag_12", "lag_336",
    "roll_mean_12", "roll_mean_48", "roll_std_48", "cluster_id",
]
feature_cols_hw = feature_cols_base + ["hw_forecast"]

spark = (
    SparkSession.builder
    .appName("DemandPredictionFeatureEngineering_GCP")
    .master("yarn")
    .config("spark.submit.deployMode", "client")
    .config("spark.eventLog.enabled", "true")
    .config("spark.executor.instances", "3")
    .config("spark.executor.cores", "3")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.memoryOverhead", "1g")
    .config("spark.driver.memory", "3g")
    .config("spark.driver.memoryOverhead", "1g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.parquet.mergeSchema", "true")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .getOrCreate()
 )
spark.sparkContext.setLogLevel("WARN")

df_all = spark.read.parquet(FEATURE_PATH)
train_df = df_all.filter(F.col("split") == "train").cache()
val_df = df_all.filter(F.col("split") == "val").cache()
test_df = df_all.filter(F.col("split") == "test").cache()
print("Train/Val/Test rows:", train_df.count(), val_df.count(), test_df.count())


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/25 15:09:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/25 15:09:30 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


Train/Val/Test rows: 19112736 2624740 5680800


In [2]:
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")
mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
r2_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="r2")

def add_smape(df):
    denom = (F.abs(F.col(TARGET_COL)) + F.abs(F.col("prediction")))
    smape = F.when(denom == 0, F.lit(0.0)).otherwise(200.0 * F.abs(F.col(TARGET_COL) - F.col("prediction")) / denom)
    return df.withColumn("smape", smape)

def evaluate(model_name, pred_df):
    pred_df = pred_df.withColumn("prediction", F.when(F.col("prediction") < 0, 0.0).otherwise(F.col("prediction")))
    rmse = float(rmse_eval.evaluate(pred_df))
    mae = float(mae_eval.evaluate(pred_df))
    r2 = float(r2_eval.evaluate(pred_df))
    mape = float(pred_df.agg(F.avg(F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)) * 100.0).first()[0])
    smape = float(add_smape(pred_df).agg(F.avg(F.col("smape"))).first()[0])
    return {"model": model_name, "RMSE": rmse, "MAE": mae, "MAPE": mape, "sMAPE": smape, "R2": r2}

assembler_base = VectorAssembler(inputCols=feature_cols_base, outputCol="features", handleInvalid="skip")
assembler_hw = VectorAssembler(inputCols=feature_cols_hw, outputCol="features", handleInvalid="skip")

model_setups = {
    "xgb_base": {"model": SparkXGBRegressor(features_col="features", label_col=TARGET_COL, prediction_col="prediction", n_estimators=80, max_depth=6, learning_rate=0.05, subsample=0.8, num_workers=3), "assembler": assembler_base},
    "xgb_hw": {"model": SparkXGBRegressor(features_col="features", label_col=TARGET_COL, prediction_col="prediction", n_estimators=80, max_depth=6, learning_rate=0.05, subsample=0.8, num_workers=3), "assembler": assembler_hw}
}

metrics_rows = []
pred_union = None
fitted_models = {}

for name, setup in model_setups.items():
    pipeline = Pipeline(stages=[setup["assembler"], setup["model"]])
    fitted = pipeline.fit(train_df)
    fitted_models[name] = fitted
    val_pred = fitted.transform(val_df)
    test_pred = fitted.transform(test_df)
    val_metrics = evaluate(name + "_val", val_pred)
    test_metrics = evaluate(name + "_test", test_pred)
    metrics_rows.extend([val_metrics, test_metrics])
    slim_pred = test_pred.select(F.lit(name).alias("model"), "PULocationID", "pickup_bin_30m", TARGET_COL, "prediction")
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)
    print("Finished model:", name)


2026-05-25 15:11:06,542 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 80}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
[15:11:55] [0]	training-rmse:22.56971                               (0 + 3) / 3]
[15:11:56] [1]	training-rmse:21.55846
[15:11:57] [2]	training-rmse:20.60184
[15:11:57] [3]	training-rmse:19.69865
[15:11:58] [4]	training-rmse:18.84454
[15:11:59] [5]	training-rmse:18.03910
[15:12:00] [6]	training-rmse:17.27705
[15:12:01] [7]	training-rmse:16.55888
[15:12:01] [8]	training-rmse:15.88122
[15:12:02] [9]	training-rmse:15.24475
[15:12:03] [10]	training-rmse:14.64430
[15:12:04] [11]	training-rmse:14.07811
[15:12:04] [12]	training-rmse:13.54665
[15:12:05] [13]	training-rmse:13.04592
[15:12:06] [14]	training-rmse:12.57784
[15:12:07] [15]	training-rmse:1

Finished model: xgb_base


2026-05-25 15:14:00,581 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 80}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
[15:14:49] [0]	training-rmse:22.56992                               (0 + 3) / 3]
[15:14:49] [1]	training-rmse:21.55812
[15:14:50] [2]	training-rmse:20.60163
[15:14:51] [3]	training-rmse:19.69819
[15:14:52] [4]	training-rmse:18.84354
[15:14:52] [5]	training-rmse:18.03848
[15:14:53] [6]	training-rmse:17.27572
[15:14:54] [7]	training-rmse:16.55801
[15:14:55] [8]	training-rmse:15.88224
[15:14:55] [9]	training-rmse:15.24378
[15:14:56] [10]	training-rmse:14.64277
[15:14:57] [11]	training-rmse:14.07867
[15:14:57] [12]	training-rmse:13.54831
[15:14:58] [13]	training-rmse:13.04959
[15:14:59] [14]	training-rmse:12.57952
[15:15:00] [15]	training-rmse:1

Finished model: xgb_hw


In [3]:
metrics_pdf = pd.DataFrame(metrics_rows)
display(metrics_pdf)

val_rows = metrics_pdf[metrics_pdf["model"].str.endswith("_val")].copy()
best_val = val_rows.sort_values("sMAPE").iloc[0]["model"].replace("_val", "")
print("Best model by val sMAPE:", best_val)

run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_BASE = f"{OUT_BASE_ROOT}/run_{run_id}"
MODEL_PATH = f"{MODEL_BASE}/run_{run_id}/{best_val}"

metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")
fitted_models[best_val].write().overwrite().save(MODEL_PATH)

print("Saved:")
print("-", f"{OUT_BASE}/metrics")
print("-", f"{OUT_BASE}/predictions")
print("-", MODEL_PATH, "(Spark ML format)")
import os
# Lấy mô hình native XGBoost từ pipeline tốt nhất (stage cuối cùng là XGBoost Model)
best_pipeline_model = fitted_models[best_val]
xgb_spark_model = best_pipeline_model.stages[-1]
native_booster = xgb_spark_model.get_booster()

# Lưu native ra file tạm local trước
local_tmp = f"/tmp/{best_val}_native.json"
native_booster.save_model(local_tmp)

# Đưa lên HDFS
native_hdfs_path = f"{MODEL_PATH}/native_booster.json"
os.system(f"hdfs dfs -mkdir -p {MODEL_PATH}")
os.system(f"hdfs dfs -put -f {local_tmp} {native_hdfs_path}")
print("-", native_hdfs_path, "(Native XGBoost format)")


,model,RMSE,MAE,MAPE,sMAPE,R2
0,xgb_base_val,7.407700,2.394932,52.948348,145.265779,0.925523
1,xgb_base_test,8.540310,2.864293,53.275811,126.491863,0.919919
2,xgb_hw_val,7.488658,2.425497,53.826723,145.220561,0.923886
3,xgb_hw_test,8.660253,2.918340,54.996431,126.552252,0.917654


/tmp/ipykernel_55263/3864443059.py:8: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")


Best model by val sMAPE: xgb_hw


26/05/25 15:16:57 WARN DAGScheduler: Broadcasting large task binary with size 1106.1 KiB


Saved:
- /user/tiennd3886/results/sparkml/run_20260525_151654/metrics
- /user/tiennd3886/results/sparkml/run_20260525_151654/predictions
- /user/tiennd3886/models/sparkml/run_20260525_151654/xgb_hw (Spark ML format)
- /user/tiennd3886/models/sparkml/run_20260525_151654/xgb_hw/native_booster.json (Native XGBoost format)


In [4]:
spark.catalog.clearCache()
spark.stop()
